In [ ]:
import glob
import numpy as np
import pandas as pd
import pandasql as ps
import polars as pl
import math
import itertools 

from xgboost import XGBClassifier
XGBOOST_AVAILABLE = True
from sklearn.model_selection import train_test_split

#matplotlib libraries
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors
import seaborn as sns

#date libraries
from dateutil import parser
from datetime import datetime, timedelta, date


#p

#pandas options
pd.set_option('display.float_format', lambda x: '%.2f' % x)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)  

pl.Config.set_tbl_cols(-1)     # show all columns, no truncation
pl.Config.set_tbl_rows(20)     # cap rows shown (avoid dumping 7M rows)
pl.Config.set_fmt_str_lengths(50)

#matplotlib setting defaults
sns.set(font="Arial",
        rc={
 "axes.axisbelow": False,
 "axes.edgecolor": "lightgrey",
 "axes.facecolor": "None",
 "axes.grid": False,
 "axes.labelcolor": "dimgrey",
 "axes.spines.right": False,
 "axes.spines.top": False,
 "figure.facecolor": "white",
 "lines.solid_capstyle": "round",
 "patch.edgecolor": "w",
 "patch.force_edgecolor": True,
 "text.color": "dimgrey",
 "xtick.bottom": False,
 "xtick.color": "dimgrey",
 "xtick.direction": "out",
 "xtick.top": False,
 "ytick.color": "dimgrey",
 "ytick.direction": "out",
 "ytick.left": False,
 "ytick.right": False})

In [ ]:
def missing_data(input_data: pl.DataFrame) -> pl.DataFrame:
    '''
    Returns a dataframe with % nulls and dtype per column.
    input: polars df
    output: polars df
    '''
    total = input_data.null_count().to_pandas().T.rename(columns={0: "Total"})
    total["Percent"] = total["Total"] / input_data.height * 100
    total["Types"] = [str(dt) for dt in input_data.dtypes]
    return total

def mape(actual, pred):
    '''
    Mean Absolute Percentage Error (MAPE)
    input: array-like actual and predicted values
    output: mape value
    '''
    actual, pred = np.array(actual), np.array(pred)
    return np.mean(np.abs((actual - pred) / actual)) * 100

In [3]:
air_data = glob.glob("bts_raw_data/*.zip")

In [4]:
import io
import zipfile

frames = []
for zip_path in air_data:
    with zipfile.ZipFile(zip_path) as archive:
        csv_name = next(name for name in archive.namelist() if name.lower().endswith('.csv'))
        with archive.open(csv_name) as csv_file:
            frames.append(pl.read_csv(io.BytesIO(csv_file.read())))

air_data = pl.concat(frames, how="diagonal_relaxed")

In [5]:
      # just the column names, cleanest check
print(air_data.describe())

shape: (9, 111)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ sta ┆ Yea ┆ Qua ┆ Mon ┆ Day ┆ Day ┆ Fli ┆ Rep ┆ DOT ┆ IAT ┆ Tai ┆ Fli ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Des ┆ Des ┆ Des ┆ Des ┆ Des ┆ Des ┆ Des ┆ Des ┆ Des ┆ CRS ┆ Dep ┆ Dep ┆ Dep ┆ Dep ┆ Dep ┆ Dep ┆ Tax ┆ Whe ┆ Whe ┆ Tax ┆ CRS ┆ Arr ┆ Arr ┆ Arr ┆ Arr ┆ Arr ┆ Arr ┆ Can ┆ Can ┆ Div ┆ CRS ┆ Ac

In [6]:
print(list(air_data.columns))

['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'IATA_CODE_Reporting_Airline', 'Tail_Number', 'Flight_Number_Reporting_Airline', 'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac', 'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'Dest', 'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'ArrivalDelayGroups', 'ArrTimeBlk', 'Cancelled', 'CancellationCode', 'Diverted', 'CRSElapsedTime', 'ActualElapsedTime', 'AirTime', 'Flights', 'Distance', 'DistanceGroup', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay', 'FirstDepTime', 'TotalAddGTime'

In [7]:
print(air_data.describe())

shape: (9, 111)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ sta ┆ Yea ┆ Qua ┆ Mon ┆ Day ┆ Day ┆ Fli ┆ Rep ┆ DOT ┆ IAT ┆ Tai ┆ Fli ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Des ┆ Des ┆ Des ┆ Des ┆ Des ┆ Des ┆ Des ┆ Des ┆ Des ┆ CRS ┆ Dep ┆ Dep ┆ Dep ┆ Dep ┆ Dep ┆ Dep ┆ Tax ┆ Whe ┆ Whe ┆ Tax ┆ CRS ┆ Arr ┆ Arr ┆ Arr ┆ Arr ┆ Arr ┆ Arr ┆ Can ┆ Can ┆ Div ┆ CRS ┆ Ac

In [8]:
feature_cols = [
    'Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek',
    'Reporting_Airline', 'Flight_Number_Reporting_Airline', 'Tail_Number',
    'Origin', 'OriginState', 'Dest', 'DestState',
    'CRSDepTime', 'CRSArrTime', 'DepTimeBlk', 'ArrTimeBlk',
    'CRSElapsedTime', 'Distance', 'DistanceGroup', 'WheelsOff', 'WheelsOn'
]

#Creating a dataframe with the features and target variable for a structured manipulatable way of messing with data
df = air_data.select(feature_cols + ['DepDel15', 'Cancelled']).to_pandas()

#DepDel15 0=Not delayed 1 delayed NaN means it was canceld. so for cancelations we simply giv ea value 2 for canceled creating a new target variable
def label(row):
    if row['Cancelled'] == 1:
        return 2
    return int(row['DepDel15'])

categorical_cols = [
    'Reporting_Airline',
    'Tail_Number',
    'Origin',
    'OriginState',
    'Dest',
    'DestState',
    'DepTimeBlk',
    'ArrTimeBlk','WheelsOn', 'WheelsOff'
]

#.apply keeps it looping through every row, we call label, and axis=1 means we are applying it to rows, not columns
y = df.apply(label, axis=1)
X = df[feature_cols].copy()


for col in categorical_cols:
    X[col] = X[col].astype('category')
#Mental list of targets we want to predict, we can add more later if we want to do multi-target prediction
#targets = ['DepDel15', 'ArrDel15', 'ArrDelayMinutes','DepDelayMinutes', 'Cancelled', 'Diverted']


In [9]:
air_xgb_model= XGBClassifier(random_state=1, enable_categorical=True)

In [ ]:
air_xgb_model.fit(X, y)

In [ ]:
print(air_xgb_model.predict(X.head(5)))

[0 0 0 0 0]


In [ ]:
print(y.head(5))

0    0
1    0
2    0
3    0
4    0
dtype: int64


In [ ]:
type_prediction=air_xgb_model.predict(X.head(5))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

air_xgb_model.fit(X_train, y_train)

type_prediction = air_xgb_model.predict(X_test)

NameError: name 'X' is not defined

In [ ]:
val_prediction = air_xgb_model.predict(X_test)

In [ ]:
print(mean_absolute_percentage_error(y_test, val_prediction))

TypeError: 'numpy.float64' object is not callable